# Mini-Challenge DEL: Hyperparameter Tuning und Model Evaluation

In [ ]:
# System & Basics
import os
import sys
import itertools
import numpy as np
import pandas as pd
from pathlib import Path

# Deep Learning Framework (PyTorch)
import torch
from torch.utils.data import DataLoader

# Computer Vision & Datasets
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# Visualisierung & Tracking
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Some EDA

In [ ]:
# Laden des svhn
def load_svhn_data(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),        
    ])

    train_dataset = datasets.SVHN(root='../data', split='train', download=True, transform=transform)
    test_dataset = datasets.SVHN(root='../data', split='test', download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

train_loader, test_loader = load_svhn_data()


In [ ]:
# EDA: Visualize some samples from the dataset
data_iter = iter(train_loader)
num_samples_to_show = 5

images, labels = next(data_iter)

plt.figure(figsize=(10, 2))
for i in range(num_samples_to_show):
    plt.subplot(1, num_samples_to_show, i + 1)
    plt.imshow(np.transpose(images[i].numpy(), (1, 2, 0)) * 0.5 + 0.5)
    plt.title(f'Label: {labels[i].item()}')
    plt.axis('off')
plt.show()


In [ ]:
# EDA: Klassen-Balanciertheit + Featurevergleich (RGB-Intensitäten je Klasse)

def prepare_labels(dataset):
    labels = np.array(dataset.labels)
    labels = np.where(labels == 10, 0, labels)
    return labels

def class_dist(labels, split_name="Train"):
    # Klassenverteilung
    class_counts = pd.Series(labels).value_counts().sort_index()
    class_pct = (class_counts / class_counts.sum() * 100).round(2)

    dist_df = pd.DataFrame({
        "Klasse": class_counts.index,
        "Anzahl": class_counts.values,
        "Anteil_%": class_pct.values
    })

    return dist_df, class_counts



def feature_comp(ds, labels, split_name="Train"):
    images = ds.data.astype(np.float32) / 255.0  # (N, C, H, W)

    # Featurevergleich: mittlere RGB-Intensitäten je Bild und Klasse
    mean_r = images[:, 0].mean(axis=(1, 2))
    mean_g = images[:, 1].mean(axis=(1, 2))
    mean_b = images[:, 2].mean(axis=(1, 2))
    mean_intensity = images.mean(axis=(1, 2, 3))

    feat_df = pd.DataFrame({
        "Klasse": labels,
        "Mean_R": mean_r,
        "Mean_G": mean_g,
        "Mean_B": mean_b,
        "Mean_Intensity": mean_intensity
    })

    class_feat = feat_df.groupby("Klasse")[["Mean_R", "Mean_G", "Mean_B", "Mean_Intensity"]].mean()
    print(f"\n=== {split_name}: Durchschnittliche Feature-Werte je Klasse ===")
    print(class_feat.round(4))

    # Visualisierungen
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.barplot(data=dist_df, x="Klasse", y="Anzahl", ax=axes[0], palette="viridis", hue="Anteil_%", dodge=False)
    axes[0].set_title(f"{split_name}: Klassenverteilung")
    axes[0].set_xlabel("Klasse")
    axes[0].set_ylabel("Anzahl")

    class_feat_plot = class_feat.reset_index().melt(
        id_vars="Klasse",
        value_vars=["Mean_R", "Mean_G", "Mean_B"],
        var_name="Feature",
        value_name="Mittelwert"
    )
    sns.barplot(data=class_feat_plot, x="Klasse", y="Mittelwert", hue="Feature", ax=axes[1], palette="Set2",)
    axes[1].set_title(f"{split_name}: RGB-Featurevergleich je Klasse")
    axes[1].set_xlabel("Klasse")
    axes[1].set_ylabel("Mittlere Intensität")
    axes[1].legend(title="Feature")

    plt.tight_layout()
    plt.show()


# EDA für Train und Test
train_labels = prepare_labels(train_loader.dataset)
test_labels = prepare_labels(test_loader.dataset)

for split_name, current_labels in [("Train", train_labels), ("Test", test_labels)]:
    dist_df, class_counts = class_dist(current_labels, split_name=split_name) 
    
    print(f"\n=== {split_name}: Klassenverteilung ===")
    print(dist_df.to_string(index=False))
    imbalance_ratio = class_counts.max() / class_counts.min()
    print(f"Imbalance-Ratio (max/min): {imbalance_ratio:.2f}")

feature_comp(train_loader.dataset, train_labels, split_name="Train")
feature_comp(test_loader.dataset, test_labels, split_name="Test")


In [ ]:
# Preprocessing der Daten
# calculate mean and std for normalization for each channel
def get_stats(loader):
    sum_, squared_sum, batches = 0, 0, 0
    
    for images, _ in loader:
        # images: [batch_size, 3, 32, 32]
        sum_ += torch.mean(images, dim=[0, 2, 3])
        squared_sum += torch.mean(images**2, dim=[0, 2, 3])
        batches += 1
        
    mean = sum_ / batches
    std = (squared_sum / batches - mean**2)**0.5
    return mean, std

# Berechnung
mean, std = get_stats(train_loader)
print(f"Mean: {mean}")
print(f"Std: {std}")


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_set = datasets.SVHN(root='../data', split='train', download=True, transform=transform)
test_set = datasets.SVHN(root='../data', split='test', download=True, transform=transform)
    
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

# 2. Visualisieren
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Normalisiertes Bild Anzeigen:
plt.imshow(images[0].permute(1, 2, 0)) # Permute für (H, W, C)
plt.title(f"Label: {labels[0]}")
plt.show()

## Metriken

Da der vorliegende Datensatz eine starke Klassenimbalance aufweist, ist die Accuracy als alleinige Metrik nicht aussagekräftig. Um die Modellleistung über alle Kategorien hinweg objektiv zu bewerten, fokussiere ich mich stattdessen auf den F1-Score sowie Precision und Recall.
Da es sich um ein Multiclass-Problem handelt, wird der CrossEntropyLoss als Verlustfunktion verwendet. Um sicherzustellen, dass auch minderbesetzte Klassen angemessen berücksichtigt werden, kommen bei der Evaluation die Macro-Averages (Macro F1, Macro Precision und Macro Recall) zum Einsatz. 


## 3. Basisfunktionalität

Die Basisfunktionalität wurde wie folgt umgesetzt:

1. **Konfiguration über `RunConfig`**
    - Zentrale Parameter (z. B. `learning_rate`, `batch_size`, `epochs`, `device`) werden in `baseline_config` gebündelt.

2. **Baseline-Training über `run_baseline(...)`**

3. **Experiment-Tracking mit Weights & Biases**
    - Aktiviert über `use_wandb=True` in der Config.
    - Projektname: `DEL`, Run-Name: `baseline-example`.

4. **Auswertung über `WandbRuns`**
    - lädt Runs aus W&B:
    ```python
    wandb_runs = WandbRuns(project="DEL")
    overview_df = wandb_runs.runs_overview(top_k=10)
    ```
    - Zusätzlich vergleichst du Metriken (z. B. `val/loss`) direkt zwischen Runs.

---

## Struktur

```text
src/
├── training.py
│   ├── RunConfig            # Konfigurationsdataclass für Training
│   └── run_baseline(...)    # Führt Baseline-Training + Evaluation aus
├── wandb_tools.py
│   └── WandbRuns            # Laden, Filtern, Vergleichen von W&B-Runs
├── model.py            # Modellarchitekturen
└── metrics.py           # F1, Precision, Recall (Macro)
```

In [ ]:
# Overfitting-Test: Kann das Basismodell einen einzelnen Batch auswendig lernen?
from torch.utils.data import DataLoader, TensorDataset
from src.training import FitConfig, run_baseline, RunConfig

# single batch aus dem train_loader extrahieren
batch_images, batch_labels = next(iter(train_loader))
overfit_dataset = TensorDataset(batch_images.clone(), batch_labels.clone())
overfit_loader = DataLoader(overfit_dataset, batch_size=len(overfit_dataset), shuffle=False)

overfit_config = RunConfig(
    learning_rate=5e-2,
    batch_size=len(overfit_dataset),
    epochs=100,
    use_wandb=False,
    wandb_project="DEL",
    run_name="overfit-single-batch",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_seed=42,
    tags=["overfit-check"],
)

overfit_result = run_baseline(
    config=overfit_config,
    fit_config=FitConfig(epochs=100),
    train_loader=overfit_loader,
)


In [ ]:
history = overfit_result["history"]
print(f"Final train accuracy: {history['train_accuracy'][-1]:.4f}")
print(f"Final val accuracy: {history['val_accuracy'][-1]:.4f}")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

## 4. Training des Basismodells

In diesem Abschnitt wird die geforderte Basis-Konfiguration umgesetzt:
- SGD ohne Momentum
- keine Regularisierung (weight decay = 0)
- keine Batch-Normalization
- manuelles Tuning von Lernrate und Batchsize (kein Auto-Search)
- Schätzung des statistischen Fehlers mit Cross-Validation

### 4a) Training mit SGD ohne Momentum/Regularisierung/BatchNorm (alle Trainingsdaten)

Es wird auf dem gesamten Trainingsset trainiert. Zur Lernkurven-Überwachung dient das offizielle Testset als Evaluationsset pro Epoche.

In [ ]:
from src.training import run_baseline, RunConfig, FitConfig
from src.wandb_tools import WandbRuns

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
wandb_runs = WandbRuns(project="DEL", entity="lukas-breiter-fachhochschule-nordwestschweiz-fhnw")


In [ ]:
baseline_config = RunConfig(
    learning_rate=1e-2,
    batch_size=64,
    epochs=20,
    use_wandb=True,
    val_fraction=0.1,
    wandb_project="DEL",
    run_name="Baseline-Example",
    tags=["baseline"],
    device=DEVICE,
    random_seed=SEED,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)
 
baseline_result = run_baseline(
    config=baseline_config,
    fit_config=FitConfig(epochs=baseline_config.epochs),
    train_loader=train_loader,
)

In [ ]:
baseline_example_hist = wandb_runs.history(
    run_path="lukas-breiter-fachhochschule-nordwestschweiz-fhnw/DEL/u4qgkxvz",
    keys=["val/macro_recall", "val/macro_precision", "val/macro_f1", "val/loss", "val/accuracy", "train/accuracy", "train/loss", "train/macro_recall", "train/macro_precision", "train/macro_f1"],
)

fig, axes = plt.subplots(4, 1, figsize=(15, 10))
axes[0].plot(baseline_example_hist["train/accuracy"], label="Train Accuracy")
axes[0].plot(baseline_example_hist["val/accuracy"], label="Val Accuracy")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(baseline_example_hist["train/loss"], label="Train Loss")
axes[1].plot(baseline_example_hist["val/loss"], label="Val Loss")
axes[1].set_title("Loss")
axes[1].legend()

axes[2].plot(baseline_example_hist["train/macro_precision"], label="Train Macro Precision")
axes[2].plot(baseline_example_hist["val/macro_precision"], label="Val Macro Precision")
axes[2].set_title("Macro Precision")
axes[2].legend()

axes[3].plot(baseline_example_hist["train/macro_recall"], label="Train Macro Recall")
axes[3].plot(baseline_example_hist["val/macro_recall"], label="Val Macro Recall")
axes[3].set_title("Macro Recall")
axes[3].legend()

plt.tight_layout()
plt.show()


In [ ]:
print("Finale Werte (4a):")
baseline_example_hist = baseline_example_hist.sort_values("_step")
print(f"val_loss: {baseline_example_hist['val/loss'].iloc[-1]:.4f}")
print(f"val_accuracy: {baseline_example_hist['val/accuracy'].iloc[-1]:.4f}")
print(f"val_macro_precision: {baseline_example_hist['val/macro_precision'].iloc[-1]:.4f}")
print(f"val_macro_recall: {baseline_example_hist['val/macro_recall'].iloc[-1]:.4f}")
print(f"val_macro_f1: {baseline_example_hist['val/macro_f1'].iloc[-1]:.4f}")

### 4b) Manuelles Tuning von Lernrate und Batchsize (keine automatische Suche)

Für das Tuning wird ausschließlich die etablierte Funktion `run_baseline(...)` aus `src.training` verwendet.
Es wird ein kleiner manueller Grid getestet (kein Bayes, kein AutoML).

In [ ]:
learning_rates = [5e-2, 1e-2, 5e-3]
batch_sizes = [32, 64, 128]
tuning_epochs = 20

tuning_runs = []
histories = {}

for lr, bs in itertools.product(learning_rates, batch_sizes):
    run_name = f"Baseline_Tuning-lr{lr}-bs{bs}"
    train_loader_cfg,_ = load_svhn_data(batch_size=bs)

    result = run_baseline(
        config=RunConfig(
            learning_rate=lr,
            batch_size=bs,
            epochs=tuning_epochs,
            val_fraction=0.1,
            use_wandb=True,
            wandb_project="DEL",
            run_name=run_name,
            tags=["baseline"],
            device=DEVICE,
            random_seed=SEED,
            num_workers=4,
            pin_memory=torch.cuda.is_available(),
        ),
        fit_config=FitConfig(epochs=tuning_epochs),
        train_loader=train_loader_cfg,
    )

    h = result["history"]
    histories[(lr, bs)] = h
    tuning_runs.append(
        {
            "learning_rate": lr,
            "batch_size": bs,
            "val_loss": h["val_loss"][-1],
            "val_accuracy": h["val_accuracy"][-1],
            "val_macro_precision": h["val_macro_precision"][-1],
            "val_macro_recall": h["val_macro_recall"][-1],
            "val_macro_f1": h["val_macro_f1"][-1],
        }
    )

In [ ]:
tuning_df = pd.DataFrame(tuning_runs).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
display(tuning_df)

best_cfg = tuning_df.iloc[0]
best_lr = float(best_cfg["learning_rate"])
best_bs = int(best_cfg["batch_size"])
print(f"Beste Konfiguration nach val_macro_f1: lr={best_lr}, batch_size={best_bs}")

In [ ]:
tuning_runs_list = pd.DataFrame(wandb_runs.runs_overview())

In [ ]:
tuning_results = []
filtered_ids = set(tuning_runs_list["id"]) - {"u4qgkxvz"}

for run_id in filtered_ids:
    baseline_example_hist = wandb_runs.history(
        run_path=f"lukas-breiter-fachhochschule-nordwestschweiz-fhnw/DEL/{run_id}",
        keys=["val/macro_recall", "val/macro_precision", "val/macro_f1", "val/loss", "val/accuracy", 
              "train/accuracy", "train/loss", "train/macro_recall", "train/macro_precision", "train/macro_f1"
              ],
    )

    hist_df = baseline_example_hist.copy()
    hist_df["run_id"] = run_id
    hist_df["learning_rate"] = tuning_runs_list.loc[tuning_runs_list["id"] == run_id, "name"].values[0].split("-")[1]
    hist_df["batch_size"] = tuning_runs_list.loc[tuning_runs_list["id"] == run_id, "name"].values[0].split("-")[2]
    tuning_results.append(hist_df)

tuning_result_df = pd.concat(tuning_results, ignore_index=True)
tuning_result_df.head()

In [ ]:
run_id	

def plot_comparison(ax, metric_key):
    for run_id	in tuning_result_df["run_id"].unique():
            
        values = tuning_result_df.loc[tuning_result_df["run_id"] == run_id, metric_key]
        epochs = tuning_result_df.loc[tuning_result_df["run_id"] == run_id, "_step"]
        
        # Plot on the specific axis passed to the function
        ax.plot(epochs, values, label=f"{tuning_result_df.loc[tuning_result_df['run_id'] == run_id, 'learning_rate'].iloc[0]}, {tuning_result_df.loc[tuning_result_df['run_id'] == run_id, 'batch_size'].iloc[0]}")

    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric_key)
    ax.legend()

# 2. Create the layout first
fig, axes = plt.subplots(2,1, figsize=(12, 8))

# 3. Pass the specific subplot (axes[0] or axes[1]) to the function
plot_comparison(axes[0], "val/loss")
plot_comparison(axes[1], "val/macro_f1")

plt.tight_layout()
plt.show()

Theorie:
- Zu grosse Lernraten können zu Oszillation oder Divergenz führen.
- Zu kleine Lernraten konvergieren stabiler, benötigen aber viel mehr Epochen. Sie könnten in lokalen Minima stecken bleiben.
- Kleinere Batches erzeugen noisigere Gradienten und können besser generalisieren, benötigen aber oft mehr Epochen und sind rechenintensiver.
- Grössere Batches liefern glattere Kurven, benötigen passende Lernraten..

### 4c) Schätzung des statistischen Fehlers mit Cross-Validation

Die Fold-Splits werden mit `kfold_indices(...)` aus `src.cross_validation` erzeugt.
Trainiert wird pro Fold mit `Trainer` aus `src.training` bei identischem Basismodell und SGD-Setup.

In [ ]:
import torch.optim as optim
from skorch import NeuralNetClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from src.model import BaselineCNN

X = train_set.data.astype(np.float32)

if X.ndim == 4 and X.shape[1] == 3:
    pass
elif X.ndim == 4 and X.shape[-1] == 3:
    X = X.transpose(0, 3, 1, 2)
else:
    raise ValueError(f"Unexpected SVHN data shape: {X.shape}")

X = X / 255.0
X = (X - mean.numpy()[None, :, None, None]) / std.numpy()[None, :, None, None]
y = train_labels.astype(np.int64)

learning_rate = best_lr if "best_lr" in globals() else 1e-2
batch_size = best_bs if "best_bs" in globals() else 64

net = NeuralNetClassifier(
    BaselineCNN,
    max_epochs=20,
    module__num_classes=10,
    lr=learning_rate,
    batch_size=batch_size,
    optimizer=optim.SGD,
    optimizer__momentum=0.0,
    optimizer__weight_decay=0.0,
    criterion=torch.nn.CrossEntropyLoss,
    device=DEVICE,
    train_split=None,
    verbose=0,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "f1_macro", "precision_macro", "recall_macro"]

results = cross_validate(
    net, X, y, 
    cv=cv, 
    scoring=scoring, 
    n_jobs=-1, 
    return_train_score=False
)

cv_summary = pd.DataFrame({
    "metric": ["accuracy", "f1_macro", "precision_macro", "recall_macro"],
    "mean": [results[f"test_{m}"].mean() for m in scoring],
    "std": [results[f"test_{m}"].std() for m in scoring],
    "se": [results[f"test_{m}"].std() / np.sqrt(cv.get_n_splits()) for m in scoring],
})

display(cv_summary)


In [ ]:
#Vizualize 

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=cv_summary, x="metric", y="mean", palette="Set2", hue="metric", errorbar=None)

ax.errorbar(
	x=np.arange(len(cv_summary)),
	y=cv_summary["mean"],
	yerr=cv_summary["se"],
	fmt="none",
	ecolor="black",
	capsize=4,
)

plt.title("Cross-Validation Performance")
plt.ylim(0, 1)
plt.show()